# LEGO Object Detection Training - Google Colab

**Thesis Project:** DLSU RIAL-3-2425-C7  
**Model:** YOLOv8n for LEGO brick detection  
**GPU Training:** 2-5 hours with Colab GPU

---

## 📋 Before You Start

1. **Enable GPU**: Runtime → Change runtime type → Hardware accelerator → GPU
2. **Upload datasets**: Upload your datasets folder to Google Drive
3. **Run cells in order**: Don't skip cells

---

## 1. Setup Environment

Mount Google Drive and check GPU availability.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted")

In [ ]:
# Check GPU availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Training will be very slow.")
    print("Go to: Runtime → Change runtime type → Hardware accelerator → GPU")

## 2. Install Dependencies

Install required packages (Ultralytics YOLOv8, etc.)

In [ ]:
# Install dependencies
!pip install -q ultralytics opencv-python pillow pandas matplotlib seaborn pyyaml tqdm scikit-learn xmltodict tensorboard

print("✓ All dependencies installed")

## 3. Setup Project Files

**Option A:** Clone from GitHub (if you've pushed your code)  
**Option B:** Upload project files from Google Drive

In [ ]:
import os
from pathlib import Path

# OPTION A: Clone from GitHub (uncomment if you have a repo)
# !git clone https://github.com/YOUR_USERNAME/block-coding-robot.git
# %cd block-coding-robot

# OPTION B: Copy from Google Drive
# Assumes you uploaded the project to: /content/drive/MyDrive/block-coding-robot/
PROJECT_PATH = '/content/drive/MyDrive/block-coding-robot'

# Check if project exists
if Path(PROJECT_PATH).exists():
    %cd $PROJECT_PATH
    print(f"✓ Project loaded from: {PROJECT_PATH}")
else:
    print(f"❌ Project not found at: {PROJECT_PATH}")
    print("\nPlease upload your project folder to Google Drive:")
    print("  1. Compress block-coding-robot folder (exclude datasets)")
    print("  2. Upload to Google Drive")
    print("  3. Extract in Google Drive")
    print("  4. Update PROJECT_PATH above")

## 4. Setup Datasets

Upload your datasets folder to Google Drive.

In [ ]:
# Check if datasets exist
datasets_path = Path('datasets')

if datasets_path.exists():
    print("✓ Datasets folder found")
    
    # Check subdirectories
    print("\nDataset structure:")
    for item in datasets_path.iterdir():
        if item.is_dir():
            count = len(list(item.glob('*')))
            print(f"  - {item.name}: {count} files")
else:
    print("❌ Datasets folder not found!")
    print("\nPlease upload datasets:")
    print("  1. Compress datasets folder on your local machine")
    print("  2. Upload to: /content/drive/MyDrive/block-coding-robot/datasets/")
    print("  3. Extract the archive")

## 5. Verify Setup

Check that all required files exist.

In [ ]:
# Verify required files
required_files = [
    'config.py',
    'prepare_datasets.py',
    'train.py',
    'validate.py',
    'test.py'
]

all_found = True
for file in required_files:
    if Path(file).exists():
        print(f"✓ {file}")
    else:
        print(f"❌ {file} not found")
        all_found = False

if all_found:
    print("\n✓ All required files present!")
else:
    print("\n❌ Some files are missing. Please upload the complete project.")

## 6. Prepare Datasets

Process and validate datasets (5-15 minutes).

In [ ]:
# Prepare datasets
!python prepare_datasets.py

## 7. Train Experiment 1 (Optional - Baseline)

Train on synthetic data only (1-2 hours with GPU).

In [ ]:
# Train Experiment 1
!python train.py --experiment 1

### Validate Experiment 1

In [ ]:
# Find the best model from Experiment 1
import glob

exp1_models = glob.glob('training_output/models/experiment_1_*/stage1_synthetic/weights/best.pt')
if exp1_models:
    exp1_model = exp1_models[-1]  # Get most recent
    print(f"Found model: {exp1_model}")
    !python validate.py --model $exp1_model --experiment 1
else:
    print("No Experiment 1 model found. Run training first.")

## 8. Train Experiment 2 (Recommended)

Two-stage training: synthetic baseline → real-world fine-tuning (2-4 hours with GPU).

In [ ]:
# Train Experiment 2
!python train.py --experiment 2

### Validate Experiment 2

In [ ]:
# Find the best model from Experiment 2
exp2_models = glob.glob('training_output/models/experiment_2_*/stage2_finetuned/weights/best.pt')
if exp2_models:
    exp2_model = exp2_models[-1]  # Get most recent
    print(f"Found model: {exp2_model}")
    !python validate.py --model $exp2_model --experiment 2
else:
    print("No Experiment 2 model found. Run training first.")

## 9. Test Model

Run inference on test images.

In [ ]:
# Test on a single image
if exp2_models:
    exp2_model = exp2_models[-1]
    test_image = 'datasets/images/00000.jpg'
    
    !python test.py --model $exp2_model --image $test_image --visualize --benchmark
else:
    print("No trained model found.")

## 10. View Results

Display training curves and validation results.

In [ ]:
from IPython.display import Image, display
import json

# Show training results
print("=" * 70)
print("EXPERIMENT 2 TRAINING RESULTS")
print("=" * 70)

if exp2_models:
    exp2_dir = Path(exp2_models[-1]).parent.parent
    
    # Show training curve
    results_plot = exp2_dir / 'results.png'
    if results_plot.exists():
        print("\nTraining Curves:")
        display(Image(filename=str(results_plot)))
    
    # Show confusion matrix
    confusion_matrix = exp2_dir / 'confusion_matrix_normalized.png'
    if confusion_matrix.exists():
        print("\nConfusion Matrix:")
        display(Image(filename=str(confusion_matrix)))

# Show validation results
print("\n" + "=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

metrics_files = glob.glob('training_output/results/*/metrics.json')
if metrics_files:
    with open(metrics_files[-1], 'r') as f:
        metrics = json.load(f)
    
    print(f"\nmAP@0.5:       {metrics['mAP@0.5']:.4f}")
    print(f"mAP@0.5:0.95:  {metrics['mAP@0.5:0.95']:.4f}")
    print(f"Precision:     {metrics['precision']:.4f}")
    print(f"Recall:        {metrics['recall']:.4f}")
    print(f"F1 Score:      {metrics['F1']:.4f}")
    
    # Check success criteria
    print("\n" + "=" * 70)
    print("SUCCESS CRITERIA CHECK")
    print("=" * 70)
    
    criteria = [
        ('mAP@0.5', metrics['mAP@0.5'], 0.70),
        ('Precision', metrics['precision'], 0.75),
        ('Recall', metrics['recall'], 0.70)
    ]
    
    all_passed = True
    for name, value, target in criteria:
        status = "✓ PASS" if value >= target else "✗ FAIL"
        print(f"{name:12s}: {value:.4f} / {target:.2f} [{status}]")
        if value < target:
            all_passed = False
    
    print(f"\nOverall: {'✓ ALL CRITERIA MET!' if all_passed else '✗ Some criteria not met'}")
    
    # Show per-class performance
    per_class_plot = Path(metrics_files[-1]).parent / 'per_class_performance.png'
    if per_class_plot.exists():
        print("\nPer-Class Performance:")
        display(Image(filename=str(per_class_plot)))

## 11. Export Model

Export to TFLite for ESP32-CAM deployment.

In [ ]:
# Export best model to TFLite
if exp2_models:
    exp2_model = exp2_models[-1]
    !python export_model.py --model $exp2_model --format tflite
else:
    print("No trained model found.")

## 12. Download Results

**Important:** Download your trained models before the Colab session ends!

Results are saved to Google Drive automatically, but you can also download them directly.

In [ ]:
# Create a zip file of all results
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f'lego_detection_results_{timestamp}'

# Zip the training_output folder
if Path('training_output').exists():
    shutil.make_archive(zip_name, 'zip', 'training_output')
    print(f"✓ Results zipped: {zip_name}.zip")
    print(f"  Size: {Path(f'{zip_name}.zip').stat().st_size / (1024*1024):.1f} MB")
    print("\nTo download:")
    print("  1. Find the zip file in the file browser (left sidebar)")
    print("  2. Right-click → Download")
    print("  3. Or the file is already saved in your Google Drive")
else:
    print("No training output found.")

## 📊 Summary

Your trained models are saved in:
- **Google Drive**: `/content/drive/MyDrive/block-coding-robot/training_output/`
- **Best model**: `training_output/models/experiment_2_*/stage2_finetuned/weights/best.pt`

### Next Steps:

1. ✅ Download your trained model
2. ✅ Review validation metrics
3. ✅ Test on real images
4. ✅ Export to TFLite for ESP32-CAM
5. ✅ Integrate with robotic arm

### Files to Keep:

- `best.pt` - Trained model
- `results.csv` - Training metrics
- `metrics.json` - Validation results
- All PNG graphs for thesis

---

**Good luck with your thesis! 🎓🤖**